In [0]:
select count(*)
from kagr_hbse.stage.vw_njd_seatmapgamesummary_enhanced
-- where PURCHASEPRICE <= 0 AND RESALEATP <=0
where saledate >= '2023-07-01 00:00:00.000';

select count(*)
from hbse.default.njd_rfm_dataset;
/************************ Dataset (game summary full) *******************************************/
CREATE or REPLACE TABLE hbse.default.njd_rfm_dataset AS
WITH base_data AS (
  SELECT
     d.audienceid, to_date(c.eventdate) as saledate,
      c.ledgername, count(c.seatnumber) as total_seats, sum(c.purchaseprice) as primary_cost, sum(c.resaleatp) as secondary_cost, (COALESCE(SUM(c.purchaseprice), 0) + COALESCE(SUM(c.resaleatp), 0)) as total_cost
  FROM
      kagr_hbse.stage.vw_njd_seatmapgamesummary_enhanced c
      join kagr_njd.stage.audiencemapping d on c.rawaudienceid = d.rawaudienceid
WHERE saledate >= '2023-07-01 00:00:00.000'
GROUP BY d.audienceid, to_date(c.eventdate), c.ledgername
)
SELECT
  audienceid, saledate, ledgername,
  SUM(total_seats) AS total_seats,
  CAST(SUM(total_cost) AS DOUBLE) AS total_sales
FROM base_data
GROUP BY audienceid, saledate, ledgername;




CREATE or REPLACE TABLE hbse.default.njd_rfm_dataset AS
WITH base_data AS (
  SELECT
     d.audienceid, to_date(c.saledate) as saledate,
      c.ledgername, count(c.seatnumber) as total_seats, sum(c.purchaseprice) as primary_cost, sum(c.resaleatp) as secondary_cost, (COALESCE(SUM(c.purchaseprice), 0) + COALESCE(SUM(c.resaleatp), 0)) as total_cost
  FROM
      kagr_hbse.stage.vw_njd_seatmapgamesummary_enhanced c
      join kagr_njd.stage.audiencemapping d on c.rawaudienceid = d.rawaudienceid
WHERE saledate >= '2023-07-01 00:00:00.000'
GROUP BY d.audienceid, to_date(c.saledate), c.ledgername
)
SELECT
  audienceid, saledate, ledgername,
  SUM(total_seats) AS total_seats,
  CAST(SUM(total_cost) AS DOUBLE) AS total_sales
FROM base_data
GROUP BY audienceid, saledate, ledgername;

/************************ Dataset (attribution) *******************************************/
CREATE or REPLACE TABLE hbse.default.njd_rfm_dataset AS
WITH base_data AS (
SELECT
	a.*, b.*, c.click
FROM
	(
	SELECT
        g.audienceid,
		a.sendid,
        a.subject,
		lower(a.emailname || 'Devils') AS emailname,
		a.emailname AS origemail,
		b.emailaddress,
		b.subscriberkey,
		to_date(b.eventdate) as senddate,
		senttime
	FROM
		kagr_njd.stage.sfmcsendjobs a
	JOIN kagr_njd.stage.sfmcsent b on a.sendid = b.sendid
    join kagr_njd.stage.rawaudience f on b.subscriberkey = f.sourceaccountid
    join kagr_njd.stage.audiencemapping g using (rawaudienceid)
 WHERE a.subject NOT LIKE '%Test%' AND a.subject NOT LIKE '%test%'and a.emailname LIKE '%Devils%') A
JOIN 
(
	SELECT
		DISTINCT a.eventname,
		'Single Game' AS ledgername,
		a.insertdate AS saledate,
		'opponent' AS opponent,
		tmsectionname AS sectionname,
		tmrowname AS rowname,
		CONCAT(tmsectionname,tmrowname,firstseat,lastseat) as SEATNUMBER,
		'1' AS seats,
		a.PURCHASEPRICE,
		lower(a.forwardtoemail) AS rawemail
	FROM
		kagr_njd.STAGE.ARCHTICSTICKETEXCHANGE  a
	WHERE
		a.insertdate IS NOT NULL
		AND purchaseprice > '0' and activityname= 'TE Resale' AND seasonname LIKE '%Regular%' AND seasonname NOT LIKE '%Parking%') b
ON
	a.emailaddress = b.rawemail
LEFT JOIN 
 (
	SELECT
		DISTINCT emailaddress,
		sendid,
		'1' AS click
	FROM
		kagr_njd.stage.sfmcclicks WHERE
	lower(URL) LIKE '%ticket%' OR lower(url) LIKE '%moveableink%' 
	OR lower(url) LIKE '%schedule%' OR lower(url) LIKE '%miDevils%'
	OR lower(url) LIKE '%ticketmaster%' OR lower(url) LIKE '%fevo%') c
 ON
	a.sendid = c.sendid
	AND a.emailaddress = c.emailaddress
WHERE
	date_trunc('DAY',senttime) < saledate AND dateadd(DAY,3,date_trunc('DAY',senttime))>= saledate AND saledate >= '2023-01-01 00:00:00.000'
)
SELECT
	audienceid, emailaddress, saledate,
	SUM(TRY_CAST(seats AS INT)) AS total_seats,
    SUM(TRY_CAST(purchaseprice AS DOUBLE)) AS total_sales,
	SUM(COALESCE(TRY_CAST(click AS INT), 0)) AS total_clicks
FROM base_data
GROUP BY audienceid, emailaddress, saledate;

/************************ Dataset (no attribution) - 5 mins *******************************************/
CREATE or REPLACE TABLE hbse.default.njd_rfm_dataset AS
WITH base_data AS (
  SELECT
    a.*, b.*, c.click
  FROM
    (
    SELECT
        g.audienceid,
        a.sendid,
        a.subject,
        lower(a.emailname || 'Devils') AS emailname,
        a.emailname AS origemail,
        b.emailaddress,
        b.subscriberkey,
        to_date(b.eventdate) as senddate,
        senttime
    FROM
        kagr_njd.stage.sfmcsendjobs a
        JOIN kagr_njd.stage.sfmcsent b ON a.sendid = b.sendid
        JOIN kagr_njd.stage.rawaudience f ON b.subscriberkey = f.sourceaccountid
        JOIN kagr_njd.stage.audiencemapping g USING (rawaudienceid)
    WHERE a.subject NOT LIKE '%Test%' AND a.subject NOT LIKE '%test%' AND a.emailname LIKE '%Devils%'
    ) a
  JOIN
    (
    SELECT
     audienceid as audienceid_key, to_date(c.saledate) as saledate,
      c.ledgername, c.tenure, count(c.seatnumber) as total_seats, sum(c.purchaseprice) as primary_cost, sum(c.resaleatp) as secondary_cost, 
      (COALESCE(c.purchaseprice, 0) + COALESCE(c.resaleatp, 0)) as total_cost
  from
      kagr_hbse.stage.vw_njd_seatmapgamesummary_enhanced c
      join kagr_njd.stage.audiencemapping d on c.rawaudienceid = d.rawaudienceid
  where
      compname = 'Not Comp'
      and ledgercode not in ('BKS', 'FSR', 'HSR')
  group by audienceid, ledgername, tenure, saledate, purchaseprice, resaleatp) b
  ON a.audienceid = b.audienceid_key AND a.senddate = b.saledate
  LEFT JOIN
    (
    SELECT DISTINCT emailaddress, sendid, '1' AS click
    FROM kagr_njd.stage.sfmcclicks
    WHERE url not like '%pref%' and url not like '%privacy_policy%'
    ) c
  ON a.sendid = c.sendid AND a.emailaddress = c.emailaddress
  WHERE
    saledate >= '2024-07-01 00:00:00.000'
)
SELECT
  audienceid, emailaddress, saledate,
  SUM(total_seats) AS total_seats,
  CAST(SUM(total_cost) AS DOUBLE) AS total_sales
FROM base_data
GROUP BY audienceid, emailaddress, saledate;

/*******************************************************************/

-- 6 minute run time
CREATE OR replace temporary table njd_emails_24 as select
    g.audienceid, b.subject, b.emailname, b.sendid, a.subscriberkey, a.emailaddress
from njd.stage.sfmcsent a
    left join njd.stage.sfmcsendjobs b on a.sendid = b.sendid
    join njd.stage.rawaudience f on a.subscriberkey = f.sourceaccountid
    join njd.stage.audiencemapping g using (rawaudienceid)
where date_trunc('year', senttime) >= '2024' and emailname LIKE 'Devils%';

--Compile send IDs for later filtering 
CREATE OR replace TEMPORARY TABLE NJD_emails_24_sendids AS SELECT
	DISTINCT sendid 
FROM njd_emails_24;

--Pull in click details
create or replace temporary table njd_clicks_24 as select
    '1' as clicked,
    a.sendid,
    to_date(a.eventdate) as clickdate,
    a.url, g.audienceid
from njd.stage.sfmcclicks a
    join njd.stage.rawaudience f on a.subscriberkey = f.sourceaccountid
    join njd.stage.audiencemapping g using (rawaudienceid)
where url not like '%pref%' and url not like '%privacy_policy%' and sendid IN (SELECT sendid FROM NJD_emails_24_sendids);

--Combine click and email data to get engagement information (7 minute run time) 
create or replace temporary table njd_email_combined_24 as select
    a.*, b.clickdate, ifnull(b.clicked, 0) as clicked, b.URL,
from njd_emails_24 a
    left join njd_clicks_24 b on a.audienceid = b.audienceid;

select count(*) from njd_email_combined_24 where clicked = 0;

--Grab data relating to actual sales 
create or replace temporary table njd_sales as select
    audienceid as audienceid_key, to_date(c.saledate) as sale_date,
    c. ledgername, c.tenure, count(c.seatnumber) as total_seats, sum(c.purchaseprice) AS primary_cost, sum(resaleatp) as secondary_cost
from
    HBSE.STAGE.VW_NJD_SEATMAPGAMESUMMARY_ENHANCED c
    join NJD.STAGE.AUDIENCEMAPPING d on c.rawaudienceid = d.rawaudienceid
where
    date_trunc('year', sale_date) >= '2024' and
    compname = 'Not Comp'
    and ledgercode not in ('BKS', 'FSR', 'HSR')
group by audienceid, c.ledgername, c.tenure, sale_date;

--Combine the sales information with the email engagement data (this has all emails sent thouhg -> do not need this)
create or replace temporary table njd_email_purchases as select
    a.*, b.sale_date, b.ledgername, b.tenure, b.total_seats, b.primary_cost, b.secondary_cost
from njd_email_combined_24 a
    left join njd_sales b on a.audienceid = b.audienceid_key;